# Baseline YOLO12-Small — Aceh Road Damage Detection 2026

Baseline pretrained empat kelas: `D00`, `D10`, `D20`, dan `D40`. Alur: audit dataset → train → validation → test → ZIP.

In [ ]:
# 1. Environment. Aktifkan GPU dan Internet pada Kaggle.
import json, platform, subprocess, sys, warnings, zipfile
from collections import Counter
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'], check=True)

# Ray pada environment Kaggle kadang meminta trial ID meskipun training ini
# bukan Ray Tune. Abaikan hanya warning spesifik yang tidak memengaruhi YOLO.
warnings.filterwarnings(
    'ignore',
    message=r'`get_trial_id` is meant to only be called.*',
    category=UserWarning,
    module=r'ray\.train\._internal\.session',
)

import torch
import ultralytics
from ultralytics import YOLO

WORKDIR = Path('/kaggle/working')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def log_section(title):
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Device      : {DEVICE}')
print(f'GPU         : {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'WARNING: GPU tidak aktif')

In [ ]:
# 2. Konfigurasi dan audit dataset sebelum training.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ardd26/datasets_here')
DATA_YAML = WORKDIR / 'ardd26_4class.yaml'
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42
EXPERIMENT_NAME = 'yolo12s_original_pretrained_ardd26'
RUNS_DIR = WORKDIR / 'runs'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATA_ROOT}')

def file_stems(folder, extensions):
    if not folder.is_dir():
        raise FileNotFoundError(f'Folder tidak ditemukan: {folder}')
    return {p.stem.casefold() for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions}

dataset_report = {}
for split in ('train', 'val', 'test'):
    image_dir, label_dir = (DATA_ROOT / split / name for name in ('images', 'labels'))
    image_stems = file_stems(image_dir, IMAGE_EXTENSIONS)
    label_stems = file_stems(label_dir, {'.txt'})
    if image_stems != label_stems:
        raise ValueError(f'Pairing {split} gagal: images={len(image_stems)}, labels={len(label_stems)}')
    counts, empty_labels = Counter(), 0
    for label_path in sorted(label_dir.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        empty_labels += int(not lines)
        for line_number, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'{label_path}:{line_number}: format bukan 5 kolom')
            class_id, coords = int(parts[0]), [float(v) for v in parts[1:]]
            if class_id not in range(4):
                raise ValueError(f'{label_path}:{line_number}: class ID {class_id} di luar 0..3')
            if not all(0 <= v <= 1 for v in coords) or coords[2] <= 0 or coords[3] <= 0:
                raise ValueError(f'{label_path}:{line_number}: bounding box invalid')
            counts[class_id] += 1
    dataset_report[split] = {'pairs': len(image_stems), 'objects': sum(counts.values()), 'empty_labels': empty_labels, 'classes': {CLASS_NAMES[i]: counts[i] for i in range(4)}}

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
''', encoding='utf-8')

log_section('DATASET AUDIT PASSED')
print(json.dumps(dataset_report, indent=2))
print(f'Total pairs: {sum(v["pairs"] for v in dataset_report.values())}')
print(f'Total objects: {sum(v["objects"] for v in dataset_report.values())}')
print(DATA_YAML.read_text())

In [ ]:
# 3. Training baseline YOLO12s pretrained.
log_section('TRAINING — YOLO12S PRETRAINED')
model = YOLO('yolo12s.pt')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
    device=DEVICE, workers=WORKERS, project=str(RUNS_DIR), name=EXPERIMENT_NAME,
    exist_ok=False, pretrained=True, optimizer=OPTIMIZER, lr0=LR0,
    momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False,
    patience=PATIENCE, seed=SEED, deterministic=True, plots=True, verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
if not BEST_PT.is_file():
    raise FileNotFoundError(f'best.pt tidak ditemukan: {BEST_PT}')
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')

In [ ]:
# 4. Validation dan test menggunakan best.pt.
def metric_summary(metrics):
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'map50_95_per_class': {CLASS_NAMES[i]: float(v) for i, v in enumerate(metrics.box.maps)},
        'save_dir': str(metrics.save_dir),
    }

best_model = YOLO(str(BEST_PT))
reports = {}
for split in ('val', 'test'):
    log_section(f'{split.upper()} — BEST.PT')
    metrics = best_model.val(
        data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_{split}', exist_ok=False, plots=True,
    )
    reports[split] = metric_summary(metrics)
    print(json.dumps(reports[split], indent=2))
VAL_REPORT, TEST_REPORT = reports['val'], reports['test']

In [ ]:
# 5. Ekspor konfigurasi, metrik, checkpoint, dan plot ke ZIP.
EVALUATION_REPORT = {'validation': VAL_REPORT, 'test': TEST_REPORT}
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'model': 'yolo12s.pt', 'dataset_root': str(DATA_ROOT), 'classes': CLASS_NAMES,
    'dataset_report': dataset_report, 'epochs': EPOCHS, 'imgsz': IMGSZ,
    'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER, 'lr0': LR0,
    'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive_items = [RUNS_DIR, DATA_YAML, RUN_CONFIG, EVALUATION_JSON]
    files = []
    for item in archive_items:
        files.extend([item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()])
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))

log_section('EXPERIMENT COMPLETED')
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created: {ZIP_PATH} ({len(files)} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))